In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "academy"
SILVER_SCHEMA = "lab4_silver"
TABLE_NAME = "netflix_titles"

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TABLE_NAME}"

In [0]:
silver_before_df = spark.table(SILVER_TABLE)

## Simulation of streaming data

Updating some rows

In [0]:
updated_records_df = (
    silver_before_df
        .filter(F.col("show_id").isin("s1", "s2", "s3"))
        .withColumn(
            "title",
            F.when(F.col("show_id") == "s1", F.concat(F.col("title"), F.lit(" - Updated"))
            ).otherwise(F.col("title"))
        )
        .withColumn(
            "rating",
            F.when(F.col("show_id")=="s2", F.lit("TV-MA")).otherwise(F.col("rating"))
        )
        .withColumn(
            "country",
            F.when(F.col("show_id")=="s3", F.lit("Poland")).otherwise(F.col("country"))
        )
        .withColumn("batch_updated_at", F.current_timestamp())
)

display(updated_records_df)

Creating some new rows

In [0]:
new_records_df = (
    silver_before_df
        .filter(F.col("show_id").isin("s4", "s5"))
        .withColumn(
            "show_id", F.when(F.col("show_id")=="s4", F.lit("s9001"))
                        .when(F.col("show_id")=="s5", F.lit("s9002"))
        )
        .withColumn(
            "title", F.when(F.col("show_id")=="s9001", F.lit("Lab4 Test Moviews"))
            .when(F.col("show_id")=="s9002", F.lit("Lab4 Test Series"))
        )
        .withColumn(
            "type", F.when(F.col("show_id")=="s9001", F.lit("Movie")).otherwise(F.lit("TV Show"))
            )
        .withColumn("batch_updated_at", F.current_timestamp())
)
display(new_records_df)

In [0]:
incremental_batch_df = (
    updated_records_df.unionByName(new_records_df)
)

In [0]:
display(incremental_batch_df)

In [0]:
duplicate_df = (
    incremental_batch_df.filter(F.col("show_id")=="s1")
    .withColumn("title", F.lit("Older duplicate version"))
    .withColumn("batch_updated_at", F.current_timestamp()- F.expr("INTERVAL 1 HOUR"))
)

In [0]:
batch_with_duplicate_df = (
    incremental_batch_df.unionByName(duplicate_df)
)

## Deduplication

In [0]:
from pyspark.sql.window import Window

dedup_window = (
    Window.partitionBy("show_id")
    .orderBy(F.col("batch_updated_at").desc())
)

In [0]:
deduplicated_batch_df = (
    batch_with_duplicate_df
    .withColumn(
        "_row_number", F.row_number().over(dedup_window)
    )
    .filter(F.col("_row_number")==1)
    .drop("_row_number")
)

In [0]:
display(deduplicated_batch_df)

In [0]:
merge_source_df = (
    deduplicated_batch_df.drop("batch_updated_at").withColumn("silver_updated_at", F.current_timestamp())
)

In [0]:
from delta.tables import DeltaTable

silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

In [0]:
from datetime import datetime 
business_columns = [col for col in merge_source_df.columns 
                   if col not in ["silver_created_at", "silver_updated_at"]]

update_map = {
    col: f"source.{col}"
    for col in business_columns
}

update_map["silver_updated_at"] = "source.silver_updated_at"

insert_map = {
    col: f"source.{col}"
    for col in business_columns
}

insert_map["silver_created_at"] = "source.silver_updated_at"
insert_map["silver_updated_at"] = "source.silver_updated_at"

In [0]:
(
    silver_delta.alias("target")
    .merge(
        merge_source_df.alias("source"), 
        "target.show_id = source.show_id"
    )
    .whenMatchedUpdate(set=update_map)
    .whenNotMatchedInsert(values=insert_map)
    .execute()
)

In [0]:
result_df = (
    spark.table(SILVER_TABLE).filter(
        F.col("show_id").isin("s1", "s2", "s3", "s9001", "s9002")
    ).orderBy("show_id")
)

display(result_df)